In [1]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

In [2]:
# Chunking the text.
from langchain_text_splitters import HTMLSectionSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_transformers import Html2TextTransformer

headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        all_chunks.extend(
            html_section_splitter.split_text(doc.page_content)
        )

    return all_chunks

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
)

def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)  
    coarse_chunks = text_splitter.split_documents(text_docs)
    return coarse_chunks


/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_47068/578083829.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_transformers import Html2TextTransformer


In [3]:
# Initialize collections.
EMBEDDING_MODEL = 'nomic-embed-text:latest'
uk_granular_collection = Chroma(
    collection_name='uk_granular',
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name='uk_coarse',
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL),
)
uk_coarse_collection.reset_collection()

In [4]:
# Create collections.
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_core.documents import Document
from langchain_community.document_transformers import BeautifulSoupTransformer
import tiktoken

CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200
TOKEN_LIMIT = 8000

def get_granular_chunks(docs: list[Document]):
    encoding = tiktoken.get_encoding("cl100k_base") 
    granular_chunks_raw = html_section_splitter.split_documents(docs)
    safety_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP
    )
    
    granular_chunks = []
    for chunk in granular_chunks_raw:
        token_count = len(encoding.encode(chunk.page_content))
        if token_count > TOKEN_LIMIT:
            sub_chunks = safety_splitter.split_documents([chunk])
            granular_chunks.extend(sub_chunks)
        else:
            granular_chunks.append(chunk)
            
    return granular_chunks
    
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]


bs_transformer = BeautifulSoupTransformer()
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()    
    docs = bs_transformer.transform_documents(
        docs, 
        tags_to_remove=["nav", "footer", "script", "style", "header"]
    )
    
    for doc in docs:
        if 'title' in doc.metadata:
            doc.metadata['Title'] = doc.metadata.pop('title')
        
        granular_chunks = get_granular_chunks([doc])                            
        uk_granular_collection.add_documents(documents=granular_chunks)
        
        coarse_chunks = split_docs_into_coarse_chunks([doc])
        uk_coarse_collection.add_documents(documents=coarse_chunks)

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  5.90it/s]


In [5]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)
    print('\n\n***************\n\n')

page_content='Brighton (disambiguation) (//en.wikivoyage.org/wiki/Brighton_(disambiguation)) .   Brighton is a seaside resort in East Sussex (//en.wikivoyage.org/wiki/East_Sussex) , south-eastern coast of England (//en.wikivoyage.org/wiki/England) , 76  km (47  mi) south of London (//en.wikivoyage.org/wiki/London) . In 1997, the district of Brighton merged with Hove to form the City of Brighton and Hove which was given city status in 2001.  Brighton is known for its grand Regency architecture, several landmarks in an oriental-inspired architectural style including the Grade-I Listed Pavilion, and for its large LGBT community (//en.wikivoyage.org/wiki/LGBT_travel) .  Understand [ edit ]  Brighton Seafront  Brighton was a sleepy little fishing village, then known as Brighthelmstone , until Dr Richard Russell of Lewes (//en.wikivoyage.org/wiki/Lewes) began to prescribe the use of seawater for his patients. He advocated the drinking of seawater and sea-bathing in 1750. In 1753 he erected a

In [6]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)
    print('\n\n***************\n\n')

page_content='pirates.com/) play in the Championship, England's second tier. Their home
ground is Mennaye Field, southwest edge of town. Events [ edit ] Penzance is
home to many ancient folk customs and festivals. They can be a colourful
spectacle, with costumed participants processing through the town, often
accompanied by musicians. Golowan Festival (http://www.golowanfestival.org/) .
Late June . Week-long festival every year at the end of June. The festival is
part revival of ancient midsummer customs practiced in the Penzance area (and
throughout Cornwall) and part arts festival. The two busiest days are Mazey
Day and Quay Fair Day where many streets are closed to traffic and the town
fills with tens of thousands of people for the processions, traditional dance,
and musicians from many Celtic nations. ( updated Jun 2017 ) [ dead link ]
Montol Festival (https://cornishculture.co.uk/festivals/montol/) . Montol is a
celebration of the Cornish traditions of Christmas and midwinter, hel